In [2]:
pip install chromadb

  Obtaining dependency information for chromadb from https://files.pythonhosted.org/packages/eb/ce/0f7be6e5d0feafa2cda54b12e6542afeea7dea89d2d411e14da90f8abb96/chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata
  Obtaining dependency information for build>=1.0.3 from https://files.pythonhosted.org/packages/0d/fe/6bea5c9162869c5beba5d9c8abbed835ec85bf1ec1fba05a3822325c45f3/build-1.5.0-py3-none-any.whl.metadata
  Obtaining dependency information for pydantic-settings>=2.0 from https://files.pythonhosted.org/packages/ae/8d/f1af3832f5e6eb13ba94ee809e72b8ecb5eef226d27ee0bef7d963d943c7/pydantic_settings-2.14.1-py3-none-any.whl.metadata
  Obtaining dependency information for pybase64>=1.4.1 from https://files.pythonhosted.org/packages/83/e3/507ab649d8c3512c258819c51d25c45d6e29d9ca33992593059e7b646a33/pybase64-1.4.3-cp312-cp312-win_amd64.whl.metadata
  Obtaining dependency information for uvicorn[standard]>=0.18.3 from https://files.pythonhosted.org/packages/88/fa/e1388bbcf24ef3274f45c0c1c7b501fd

In [4]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [8]:
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
"""
Step 1: Index the knowledge base into Chroma.

We load knowledge_base.json, turn each passage into an embedding vector,
and store it in a Chroma collection together with its source as metadata.

NOTE on chunking:
Each entry in knowledge_base.json is already a short, single-topic passage,
so no splitting is needed here. If this were a full document instead
(e.g. a 40-page handbook), we WOULD need to chunk it first: embedding the
whole document as one vector would average unrelated topics (onboarding,
payroll, parking, security...) into a single blurry point in meaning-space.
That makes retrieval imprecise (a query about "parking" would weakly match
everything) and forces us to stuff the entire document into the prompt
instead of just the relevant paragraph. Chunking keeps each vector focused
on one topic, so retrieval is sharp and the prompt stays lean.
"""

import json
import os

import chromadb
import google.generativeai as genai

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

EMBED_MODEL = "models/gemini-embedding-001"
JSON_PATH = "knowledge_base.json"
COLLECTION_NAME = "company_kb"
CHROMA_PATH = "./chroma_db"


def embed_text(text: str, task_type: str = "retrieval_document") -> list[float]:
    """Turn a passage of text into an embedding vector using Gemini."""
    result = genai.embed_content(model=EMBED_MODEL, content=text, task_type=task_type)
    return result["embedding"]


def get_passage_text(entry: dict, idx: int) -> str:
    """Try the common field names a passage's text might be stored under."""
    for key in ("text", "content", "passage"):
        if key in entry:
            return entry[key]
    raise ValueError(
        f"Entry {idx} has no recognizable text field (got keys: {list(entry.keys())}). "
        "Update get_passage_text() to match your JSON's actual key."
    )


def build_index(json_path: str = JSON_PATH, collection_name: str = COLLECTION_NAME):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    client = chromadb.PersistentClient(path=CHROMA_PATH)
    # Drop any previous run so re-indexing doesn't duplicate entries
    client.delete_collection(collection_name) if collection_name in [
        c.name for c in client.list_collections()
    ] else None
    collection = client.create_collection(name=collection_name)

    ids, texts, metadatas, embeddings = [], [], [], []
    for i, entry in enumerate(data):
        text = get_passage_text(entry, i)
        source = entry.get("source", f"unknown_{i}")

        ids.append(str(i))
        texts.append(text)
        metadatas.append({"source": source})
        embeddings.append(embed_text(text))

    collection.add(ids=ids, documents=texts, metadatas=metadatas, embeddings=embeddings)
    print(f"Indexed {len(ids)} passages into Chroma collection '{collection_name}'.")
    return collection


if __name__ == "__main__":
    build_index()


Indexed 10 passages into Chroma collection 'company_kb'.


In [12]:
"""
Step 2: Query — retrieve the top-k passages for a question and assemble
them into a grounded prompt (not sent to the model yet, that's step 3).
"""

import os

import chromadb
import google.generativeai as genai
from dotenv import load_dotenv

load_dotenv()
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

EMBED_MODEL = "models/gemini-embedding-001"
CHROMA_PATH = "./chroma_db"
COLLECTION_NAME = "company_kb"

PROMPT_TEMPLATE = """Answer the question using ONLY the context below.
If the context does not contain the answer, say "I don't know" — do not make up an answer.
For every fact you use, mention which source it came from.

Context:
{context_block}

Question: {question}

Answer:"""


def embed_text(text: str, task_type: str) -> list[float]:
    """task_type should be 'retrieval_document' for passages (indexing)
    and 'retrieval_query' for questions (querying) — must match the
    model used when the passages were indexed."""
    result = genai.embed_content(model=EMBED_MODEL, content=text, task_type=task_type)
    return result["embedding"]


def retrieve(question: str, top_k: int = 3):
    """Return a list of (passage_text, source) tuples for the top_k matches."""
    client = chromadb.PersistentClient(path=CHROMA_PATH)
    collection = client.get_collection(COLLECTION_NAME)

    query_embedding = embed_text(question, task_type="retrieval_query")

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
    )

    passages = results["documents"][0]
    sources = [m["source"] for m in results["metadatas"][0]]
    return list(zip(passages, sources))


def build_prompt(question: str, retrieved: list[tuple[str, str]]) -> str:
    """Assemble the retrieved passages (tagged with source) and the
    question into the final grounded prompt."""
    context_block = "\n\n".join(
        f"[Source: {source}]\n{passage}" for passage, source in retrieved
    )
    return PROMPT_TEMPLATE.format(context_block=context_block, question=question)


if __name__ == "__main__":
    question = "How long do I have to get a full refund?"
    retrieved = retrieve(question, top_k=3)

    print("Retrieved sources:")
    for passage, source in retrieved:
        print(f"  - {source}: {passage[:80]}...")

    prompt = build_prompt(question, retrieved)
    print("\n--- ASSEMBLED PROMPT ---\n")
    print(prompt)

Retrieved sources:
  - policy.md: Our refund policy allows a full refund within 30 days of purchase, provided the ...
  - policy.md: To cancel your subscription, open Account Settings and choose End Plan. Cancella...
  - policy.md: Premium plan members get priority support, with a guaranteed first response with...

--- ASSEMBLED PROMPT ---

Answer the question using ONLY the context below.
If the context does not contain the answer, say "I don't know" — do not make up an answer.
For every fact you use, mention which source it came from.

Context:
[Source: policy.md]
Our refund policy allows a full refund within 30 days of purchase, provided the item is unused and in its original packaging. After 30 days, only store credit is offered.

[Source: policy.md]
To cancel your subscription, open Account Settings and choose End Plan. Cancellation takes effect at the end of the current billing period; no partial refunds are given for the remaining days.

[Source: policy.md]
Premium plan members 

In [19]:
CHAT_MODEL = "gemini-2.5-flash"

In [20]:
# ===================== Generate =====================
def generate_answer(question: str, top_k: int = 3) -> dict:
    """Retrieve passages, build the grounded prompt, ask the chat model,
    and return the sources used plus the cited answer."""
    retrieved = retrieve(question, top_k=top_k)
    prompt = build_prompt(question, retrieved)

    model = genai.GenerativeModel(CHAT_MODEL)
    response = model.generate_content(prompt)

    return {
        "question": question,
        "sources": [source for _, source in retrieved],
        "answer": response.text,
    }


# ===================== Run the three required questions =====================
questions = [
    "How long do I have to get a full refund?",
    "How do I reset my password?",
    "What is the company's stock price today?",
]

for q in questions:
    result = generate_answer(q)
    print(f"Q: {result['question']}")
    print(f"Retrieved sources: {result['sources']}")
    print(f"Answer:\n{result['answer']}")
    print("-" * 60)

Q: How long do I have to get a full refund?
Retrieved sources: ['policy.md', 'policy.md', 'policy.md']
Answer:
You have 30 days of purchase to get a full refund, provided the item is unused and in its original packaging [Source: policy.md].
------------------------------------------------------------
Q: How do I reset my password?
Retrieved sources: ['it.md', 'policy.md', 'handbook.md']
Answer:
You can reset your password from the login screen by clicking 'Forgot password' [Source: it.md]. A reset link is emailed to your registered address [Source: it.md] and expires after one hour for security [Source: it.md].
------------------------------------------------------------
Q: What is the company's stock price today?
Retrieved sources: ['policy.md', 'policy.md', 'facilities.md']
Answer:
I don't know.
------------------------------------------------------------


## Results

### Q1: How long do I have to get a full refund?
**Retrieved sources:** policy.md, policy.md, policy.md

**Answer:** You have 30 days of purchase to get a full refund, provided the item is unused and in its original packaging [Source: policy.md].

---

### Q2: How do I reset my password?
**Retrieved sources:** it.md, policy.md, handbook.md

**Answer:** You can reset your password from the login screen by clicking 'Forgot password' [Source: it.md]. A reset link is emailed to your registered address [Source: it.md] and expires after one hour for security [Source: it.md].

---

### Q3: What is the company's stock price today?
**Retrieved sources:** policy.md, policy.md, facilities.md

**Answer:** I don't know.

*This question is out of scope for the knowledge base — the system correctly declined instead of inventing an answer.*